## Text Cleaning

In [1]:
# import libraries
import pandas as pd
import numpy as np
import nltk
import re

In [2]:
# load dataset
df = pd.read_csv('data/headlines.csv')
df = df.drop(columns=['Unnamed: 0', 'news_article'])
df = df.rename(columns={'news_headline': 'headline', 'news_category': 'category'})

print(df.head())

                                            headline    category
0  50-year-old problem of biology solved by Artif...  technology
1  Microsoft Teams to stop working on Internet Ex...  technology
2  Hope US won't erect barriers to cooperation: C...  technology
3  Global smartphone sales in Q3 falls 5.7% to 36...  technology
4  EU hoping Biden will clarify US position on di...  technology


In [3]:
# remove duplicate headlines
df = df.drop_duplicates(subset='headline').reset_index(drop=True)
print(f"Total rows: {len(df)}")
print(f"Duplicate headlines: {df['headline'].duplicated().sum()}")

Total rows: 1996
Duplicate headlines: 0


In [4]:
# check missing values
print(df.isnull().sum())

headline    0
category    0
dtype: int64


In [5]:
# lowercase 
df['headline'] = df['headline'].str.lower()
print(df.head())

                                            headline    category
0  50-year-old problem of biology solved by artif...  technology
1  microsoft teams to stop working on internet ex...  technology
2  hope us won't erect barriers to cooperation: c...  technology
3  global smartphone sales in q3 falls 5.7% to 36...  technology
4  eu hoping biden will clarify us position on di...  technology


In [6]:
# remove punctuation, numbers, and special characters
df['headline'] = df['headline'].apply(lambda headline: re.sub(r'[^A-Za-z]', ' ', headline))
print(df.head())

                                            headline    category
0     year old problem of biology solved by artif...  technology
1  microsoft teams to stop working on internet ex...  technology
2  hope us won t erect barriers to cooperation  c...  technology
3  global smartphone sales in q  falls      to   ...  technology
4  eu hoping biden will clarify us position on di...  technology


In [7]:
# tokenization
from nltk.tokenize import word_tokenize

df['headline'] = df['headline'].apply(word_tokenize)
print(df.head())

                                            headline    category
0  [year, old, problem, of, biology, solved, by, ...  technology
1  [microsoft, teams, to, stop, working, on, inte...  technology
2  [hope, us, won, t, erect, barriers, to, cooper...  technology
3  [global, smartphone, sales, in, q, falls, to, ...  technology
4  [eu, hoping, biden, will, clarify, us, positio...  technology


In [8]:
# remove stop words
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
df['headline'] = df['headline'].apply(lambda tokens: [w for w in tokens if w not in stop_words])
print(df.head())

                                            headline    category
0  [year, old, problem, biology, solved, artifici...  technology
1  [microsoft, teams, stop, working, internet, ex...  technology
2  [hope, us, erect, barriers, cooperation, china...  technology
3  [global, smartphone, sales, q, falls, cr, unit...  technology
4  [eu, hoping, biden, clarify, us, position, dig...  technology


In [9]:
# lemmatization
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

def get_wordnet_pos(pos_tag):
    first_letter = pos_tag[0]

    if first_letter == 'J':
        return wordnet.ADJ
    elif first_letter == 'V':
        return wordnet.VERB
    elif first_letter == 'R':
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
lemmatizer = WordNetLemmatizer()

df['headline'] = df['headline'].apply(lambda tokens: [ 
    lemmatizer.lemmatize(word, get_wordnet_pos(pos_tag)) 
    for (word, pos_tag) in nltk.pos_tag(tokens)
    ])
print(df.head())

                                            headline    category
0  [year, old, problem, biology, solve, artificia...  technology
1  [microsoft, team, stop, work, internet, explor...  technology
2  [hope, u, erect, barrier, cooperation, china, ...  technology
3  [global, smartphone, sale, q, fall, cr, unit, ...  technology
4  [eu, hop, biden, clarify, u, position, digital...  technology


In [10]:
# join words
df['headline'] = df['headline'].apply(lambda tokens: ' '.join(tokens))
print(df.head())

                                            headline    category
0  year old problem biology solve artificial inte...  technology
1     microsoft team stop work internet explorer nov  technology
2  hope u erect barrier cooperation china blackli...  technology
3      global smartphone sale q fall cr unit gartner  technology
4  eu hop biden clarify u position digital tax re...  technology


In [11]:
# X and y
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X = df['headline']
y = le.fit_transform(df['category'])

In [12]:
# train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

## TF-IDF

In [13]:
# tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

print(X_train[:20])

  (0, 157)	0.5094512434614435
  (0, 310)	0.41752442455522293
  (0, 881)	0.38041563969870534
  (0, 4055)	0.5094512434614435
  (0, 2268)	0.31247954660461064
  (0, 1839)	0.2534416895410255
  (1, 2607)	0.40326484351612374
  (1, 2758)	0.38198208105446435
  (1, 1789)	0.38198208105446435
  (1, 2360)	0.40326484351612374
  (1, 1403)	0.2773999183268285
  (1, 4265)	0.2647413820746047
  (1, 2652)	0.2706899765189821
  (1, 256)	0.40326484351612374
  (2, 1424)	0.3257109305412842
  (2, 4151)	0.33645221373379247
  (2, 695)	0.2168382438246373
  (2, 2858)	0.3698176868412778
  (2, 1324)	0.7006002725091574
  (2, 611)	0.3257109305412842
  (3, 2803)	0.4205443169776718
  (3, 1972)	0.4205443169776718
  (3, 578)	0.2741644116438953
  (3, 2011)	0.31402737242867074
  (3, 1092)	0.4205443169776718
  :	:
  (17, 900)	0.28255955768485586
  (17, 650)	0.38742040247388376
  (17, 3706)	0.38742040247388376
  (17, 2712)	0.26650076180049
  (17, 133)	0.3524667875442078
  (17, 1404)	0.34121423688047076
  (17, 20)	0.341214236880

## Logistic Regression

In [14]:
# fit logistic regression
from sklearn.linear_model import LogisticRegression

log = LogisticRegression()
log.fit(X_train, y_train)

LogisticRegression()

In [15]:
# predict X_test
y_pred_log = log.predict(X_test)

print('(Actual, Predicted)')
print(list(zip(y_test, y_pred_log)))

(Actual, Predicted)
[(5, 5), (2, 6), (1, 1), (2, 2), (1, 1), (1, 1), (4, 1), (5, 5), (6, 6), (5, 5), (3, 1), (6, 6), (4, 1), (4, 4), (3, 6), (6, 6), (6, 6), (5, 6), (4, 4), (3, 6), (1, 1), (1, 1), (3, 6), (6, 1), (1, 1), (5, 5), (2, 2), (4, 4), (3, 5), (6, 6), (4, 4), (3, 5), (6, 6), (2, 2), (6, 1), (4, 4), (4, 1), (3, 1), (1, 1), (5, 5), (4, 4), (3, 1), (0, 5), (5, 5), (6, 6), (4, 4), (6, 6), (6, 6), (1, 1), (6, 6), (1, 1), (1, 1), (2, 1), (4, 4), (6, 6), (5, 5), (2, 2), (5, 5), (1, 1), (0, 6), (3, 5), (2, 2), (5, 5), (6, 6), (4, 1), (1, 1), (1, 1), (0, 5), (0, 0), (4, 4), (6, 6), (2, 2), (6, 6), (4, 4), (1, 1), (0, 6), (0, 5), (4, 6), (1, 1), (2, 2), (6, 6), (1, 2), (1, 4), (4, 4), (5, 1), (4, 4), (1, 1), (4, 4), (1, 1), (2, 6), (6, 6), (4, 4), (3, 6), (5, 5), (1, 1), (1, 1), (6, 6), (2, 1), (2, 1), (4, 4), (6, 6), (4, 4), (4, 6), (6, 6), (4, 4), (6, 6), (4, 1), (5, 5), (4, 4), (4, 4), (1, 1), (4, 4), (2, 2), (3, 4), (1, 1), (1, 4), (4, 4), (2, 2), (3, 6), (5, 5), (6, 6), (5, 5), (5,

In [16]:
# evaluate logistic regression
from sklearn.metrics import accuracy_score, confusion_matrix

acc_log = accuracy_score(y_test, y_pred_log)
cm_log = confusion_matrix(y_test, y_pred_log)

print(f'Accuracy: {acc_log:.3f}\n')
print(f'Confusion Matrix:\n{cm_log}')

Accuracy: 0.733

Confusion Matrix:
[[ 3  1  0  0  2  6  6]
 [ 0 75  1  0  6  1  3]
 [ 0  9 40  0  0  0  5]
 [ 0  5  0  3  2  6 10]
 [ 0 12  0  0 62  0  4]
 [ 1  4  0  0  2 39  9]
 [ 0  9  2  0  0  1 71]]


In [17]:
# save model and vectorizer
import joblib

joblib.dump(log, 'model/classifier.pkl')
joblib.dump(vectorizer, 'model/vectorizer.pkl')

['model/vectorizer.pkl']